# Problem 01: Academic Audit & Credit-Weighted CPI Performance Engine

## Design Approach

A dictionary is used as the master data structure because each student can
be accessed efficiently using their unique StudentID as the key.

Each student's information is stored as:
StudentID -> {"name": str, "courses": list}

The courses are stored in a list because a student can complete multiple
courses. Each individual course is represented using an immutable tuple:

(CourseCode, Credits, LetterGrade)

A dictionary is used to map letter grades to grade points:

A -> 10
B -> 8
C -> 6
D -> 4
F -> 0

The CPI is calculated using the credit-weighted formula:

CPI = Sum(Credits × GradePoint) / Sum(Credits)

The merit list is sorted using three criteria:
1. CPI in descending order.
2. Total credits in descending order when CPI is equal.
3. Student name in alphabetical order when both CPI and credits are equal.

## Complexity Analysis

Let N be the number of students and C be the maximum number of courses
per student.

- Storing student records: O(N × C)
- Calculating CPI: O(N × C)
- Sorting the merit list: O(N log N)
- Overall time complexity: O(N × C + N log N)
- Space complexity: O(N × C)

In [2]:
from typing import Dict, List, Tuple


# Type alias for a course record
Course = Tuple[str, int, str]

# Type alias for a student record
Student = Dict[str, object]

# Grade-to-point mapping
GRADE_POINTS: Dict[str, int] = {
    "A": 10,
    "B": 8,
    "C": 6,
    "D": 4,
    "F": 0,
}


def calculate_student_performance(
    courses: List[Course],
) -> Tuple[int, float]:
    """
    Calculate total credits and credit-weighted CPI for a student.

    Args:
        courses: List of course tuples containing
                 (course_code, credits, letter_grade).

    Returns:
        A tuple containing total credits and CPI rounded to 2 decimals.
    """
    total_credits = 0
    weighted_points = 0

    for course_code, credits, grade in courses:
        total_credits += credits
        weighted_points += credits * GRADE_POINTS[grade]

    cpi = weighted_points / total_credits

    return total_credits, round(cpi, 2)


def create_merit_list(
    students: Dict[str, Student],
) -> List[Tuple[str, Student, int, float]]:
    """
    Create the department merit list.

    Sorting criteria:
    1. CPI descending
    2. Total credits descending
    3. Student name alphabetically
    """
    performance_list = []

    for student_id, student in students.items():
        courses = student["courses"]
        total_credits, cpi = calculate_student_performance(courses)

        performance_list.append(
            (student_id, student, total_credits, cpi)
        )

    performance_list.sort(
        key=lambda item: (
            -item[3],              # CPI descending
            -item[2],              # Credits descending
            item[1]["name"],       # Name ascending
        )
    )

    return performance_list


def process_input(input_data: str) -> None:
    """
    Read student academic records from input and print the required output.
    """
    lines = input_data.strip().splitlines()
    index = 0

    number_of_students = int(lines[index])
    index += 1

    students: Dict[str, Student] = {}

    for _ in range(number_of_students):
        student_id, name = lines[index].split()
        index += 1

        number_of_courses = int(lines[index])
        index += 1

        courses: List[Course] = []

        for _ in range(number_of_courses):
            course_code, credits, grade = lines[index].split()
            index += 1

            courses.append(
                (course_code, int(credits), grade)
            )

        students[student_id] = {
            "name": name,
            "courses": courses,
        }

    # Print students in original input order.
    for student_id, student in students.items():
        total_credits, cpi = calculate_student_performance(
            student["courses"]
        )

        print(
            f"{student_id} {student['name']} "
            f"{total_credits} {cpi:.2f}"
        )

    print("\n-- Department Merit List --")

    merit_list = create_merit_list(students)

    for rank, (_, student, total_credits, cpi) in enumerate(
        merit_list, start=1
    ):
        print(
            f"Rank {rank}: {student['name']} "
            f"(CPI: {cpi:.2f}, Credits: {total_credits})"
        )


# Sample input
sample_input = """\
3
S101 Alice
3
CS101 4 A
MA101 4 B
HS101 2 A
S102 Bob
2
CS101 4 B
MA101 4 C
S103 Charlie
3
CS101 4 A
MA101 4 A
HS101 2 B
"""

process_input(sample_input)

S101 Alice 10 9.20
S102 Bob 8 7.00
S103 Charlie 10 9.60

-- Department Merit List --
Rank 1: Charlie (CPI: 9.60, Credits: 10)
Rank 2: Alice (CPI: 9.20, Credits: 10)
Rank 3: Bob (CPI: 7.00, Credits: 8)


In [3]:
# Verification Test 1: Provided sample
print("===== TEST 1: PROVIDED SAMPLE =====")
process_input(sample_input)


# Verification Test 2: Boundary case with one student
boundary_input = """\
1
S001 Rahul
1
CS101 1 F
"""

print("\n===== TEST 2: BOUNDARY CASE =====")
process_input(boundary_input)

===== TEST 1: PROVIDED SAMPLE =====
S101 Alice 10 9.20
S102 Bob 8 7.00
S103 Charlie 10 9.60

-- Department Merit List --
Rank 1: Charlie (CPI: 9.60, Credits: 10)
Rank 2: Alice (CPI: 9.20, Credits: 10)
Rank 3: Bob (CPI: 7.00, Credits: 8)

===== TEST 2: BOUNDARY CASE =====
S001 Rahul 1 0.00

-- Department Merit List --
Rank 1: Rahul (CPI: 0.00, Credits: 1)


# Problem 02: Library Circulation & Borrower Cohort Jaccard Similarity

## Design Approach

A dictionary is used for the book catalog because each BookID is unique.
The dictionary maps:

BookID -> (Title, Author)

The title and author are stored as an immutable tuple.

Two sets are maintained:
- available_books: contains BookIDs currently available.
- borrowed_books: contains BookIDs currently borrowed.

Another dictionary maps each StudentID to a set of BookIDs currently
borrowed by that student.

Sets are useful because checking membership, adding, and removing a book
can be performed in average O(1) time.

For COMMON, set intersection is used.

For SIMILARITY, Jaccard Similarity is calculated as:

J(S1, S2) = |S1 ∩ S2| / |S1 ∪ S2|

If both students have no borrowed books, the similarity is 0.00.

## Complexity Analysis

Let B be the number of books and Q be the number of operations.

- Catalog construction: O(B)
- BORROW: O(1) average
- RETURN: O(1) average
- COMMON: O(min(S1, S2)) average
- SIMILARITY: O(min(S1, S2)) average
- INVENTORY: O(1)
- Total space complexity: O(B + U + R)

where U is the number of students and R is the number of active
student-book relationships.

In [4]:
from typing import Dict, Set, Tuple


# Book information is stored as an immutable tuple.
Book = Tuple[str, str]


def process_library(input_data: str) -> None:
    """
    Process library catalog and circulation operations.
    """
    lines = input_data.strip().splitlines()
    index = 0

    # Read number of books.
    number_of_books = int(lines[index])
    index += 1

    # BookID -> (Title, Author)
    catalog: Dict[int, Book] = {}

    # Initially, every catalog book is available.
    available_books: Set[int] = set()

    # Books currently borrowed.
    borrowed_books: Set[int] = set()

    # StudentID -> set of borrowed BookIDs.
    student_books: Dict[str, Set[int]] = {}

    # Build the catalog.
    for _ in range(number_of_books):
        book_id, title, author = lines[index].split()
        index += 1

        book_id = int(book_id)

        catalog[book_id] = (title, author)
        available_books.add(book_id)

    # Read number of operations.
    number_of_operations = int(lines[index])
    index += 1

    for _ in range(number_of_operations):
        operation = lines[index].split()
        index += 1

        command = operation[0]

        if command == "BORROW":
            student_id = operation[1]
            book_id = int(operation[2])

            # Check whether the book exists.
            if book_id not in catalog:
                print("Book Not Found")

            # Check whether the book is available.
            elif book_id not in available_books:
                print("Already Borrowed")

            else:
                # Create the student's set if necessary.
                if student_id not in student_books:
                    student_books[student_id] = set()

                student_books[student_id].add(book_id)
                available_books.remove(book_id)
                borrowed_books.add(book_id)

                print("Borrow Success")

        elif command == "RETURN":
            student_id = operation[1]
            book_id = int(operation[2])

            # Check whether the student currently has this book.
            if (
                student_id not in student_books
                or book_id not in student_books[student_id]
            ):
                print("Return Error")

            else:
                student_books[student_id].remove(book_id)
                available_books.add(book_id)
                borrowed_books.remove(book_id)

                print("Return Success")

        elif command == "COMMON":
            student_1 = operation[1]
            student_2 = operation[2]

            books_1 = student_books.get(student_1, set())
            books_2 = student_books.get(student_2, set())

            common_books = books_1 & books_2

            if not common_books:
                print("None")
            else:
                print(
                    " ".join(
                        str(book_id)
                        for book_id in sorted(common_books)
                    )
                )

        elif command == "SIMILARITY":
            student_1 = operation[1]
            student_2 = operation[2]

            books_1 = student_books.get(student_1, set())
            books_2 = student_books.get(student_2, set())

            intersection = books_1 & books_2
            union = books_1 | books_2

            if not books_1 and not books_2:
                similarity = 0.00
            else:
                similarity = len(intersection) / len(union)

            print(f"Similarity: {similarity:.2f}")

        elif command == "INVENTORY":
            print(
                f"Available: {len(available_books)}, "
                f"Borrowed: {len(borrowed_books)}"
            )


# Provided sample input
sample_input_2 = """\
4
101 Algorithms Cormen
102 PythonCookbook Mark
103 DatabaseSystems Korth
104 OperatingSystems Silberschatz
7
BORROW S01 101
BORROW S01 102
BORROW S02 102
BORROW S02 103
COMMON S01 S02
SIMILARITY S01 S02
INVENTORY
"""

process_library(sample_input_2)

Borrow Success
Borrow Success
Already Borrowed
Borrow Success
None
Similarity: 0.00
Available: 1, Borrowed: 3


In [5]:
# Verification Test 1: Provided sample
print("===== TEST 1: PROVIDED SAMPLE =====")
process_library(sample_input_2)


# Verification Test 2: Return and common-book check
test_input = """\
3
101 Algorithms Cormen
102 PythonCookbook Mark
103 DatabaseSystems Korth
6
BORROW S01 101
BORROW S02 101
RETURN S01 101
BORROW S02 102
COMMON S01 S02
INVENTORY
"""

print("\n===== TEST 2: RETURN TEST =====")
process_library(test_input)


# Verification Test 3: Both students have no books
empty_similarity_input = """\
1
101 Algorithms Cormen
1
SIMILARITY S01 S02
"""

print("\n===== TEST 3: EMPTY PORTFOLIOS =====")
process_library(empty_similarity_input)

===== TEST 1: PROVIDED SAMPLE =====
Borrow Success
Borrow Success
Already Borrowed
Borrow Success
None
Similarity: 0.00
Available: 1, Borrowed: 3

===== TEST 2: RETURN TEST =====
Borrow Success
Already Borrowed
Return Success
Borrow Success
None
Available: 2, Borrowed: 1

===== TEST 3: EMPTY PORTFOLIOS =====
Similarity: 0.00


# Problem 03: Warehouse Inventory, Tag Matching & Atomic Order Processing

## Design Approach

A dictionary is used as the main product catalog:

ProductID -> {
    "name": str,
    "price": float,
    "stock": int,
    "tags": set
}

A set is used for product tags because set operations such as issubset()
and intersection() allow efficient tag matching.

Another set, OutOfStockIDs, stores the IDs of products whose stock is zero.
This allows quick availability checking.

Customer orders are represented as a list of tuples:

[(ProductID, RequestedQuantity), ...]

For FILTER_AND, issubset() is used to find products containing all
requested tags.

For FILTER_OR, set intersection is used to find products containing
at least one requested tag.

For ORDER, atomic transaction processing is used. First, every requested
product is checked for existence and sufficient stock. Stock is changed
only after all items pass validation. Therefore, a failed order cannot
partially modify inventory.

## Complexity Analysis

Let P be the number of products, Q be the number of operations, and K be
the number of items in an order.

- Product insertion: O(1) average per product
- FILTER_AND: O(P × T) in the straightforward implementation
- FILTER_OR: O(P × T) in the straightforward implementation
- Order validation: O(K) average
- Order stock update: O(K)
- Restock: O(1) average
- Space complexity: O(P × T + K)

where T is the number of tags associated with a product.

In [6]:
from typing import Dict, List, Set, Tuple


# Product information type.
Product = Dict[str, object]

# Order item: (ProductID, RequestedQuantity)
OrderItem = Tuple[int, int]


def process_warehouse(input_data: str) -> None:
    """
    Process warehouse inventory and customer operations.
    """
    lines = input_data.strip().splitlines()
    index = 0

    # Number of products.
    number_of_products = int(lines[index])
    index += 1

    # ProductID -> product information.
    products: Dict[int, Product] = {}

    # Products whose stock is currently zero.
    out_of_stock_ids: Set[int] = set()

    # Read product information.
    for _ in range(number_of_products):
        parts = lines[index].split()
        index += 1

        product_id = int(parts[0])
        name = parts[1]
        price = float(parts[2])
        stock = int(parts[3])
        tag_count = int(parts[4])

        tags = set(parts[5:5 + tag_count])

        products[product_id] = {
            "name": name,
            "price": price,
            "stock": stock,
            "tags": tags,
        }

        if stock == 0:
            out_of_stock_ids.add(product_id)

    # Number of operations.
    number_of_operations = int(lines[index])
    index += 1

    for _ in range(number_of_operations):
        parts = lines[index].split()
        index += 1

        command = parts[0]

        # ---------------------------------------------------------
        # FILTER_AND
        # ---------------------------------------------------------
        if command == "FILTER_AND":
            query_tags = set(parts[1:])
            matching_products: List[int] = []

            for product_id, product in products.items():
                product_tags = product["tags"]

                if query_tags.issubset(product_tags):
                    matching_products.append(product_id)

            matching_products.sort()

            if matching_products:
                print(" ".join(map(str, matching_products)))
            else:
                print("No Match")

        # ---------------------------------------------------------
        # FILTER_OR
        # ---------------------------------------------------------
        elif command == "FILTER_OR":
            query_tags = set(parts[1:])
            matching_products = []

            for product_id, product in products.items():
                product_tags = product["tags"]

                if product_tags.intersection(query_tags):
                    matching_products.append(product_id)

            matching_products.sort()

            if matching_products:
                print(" ".join(map(str, matching_products)))
            else:
                print("No Match")

        # ---------------------------------------------------------
        # ORDER
        # ---------------------------------------------------------
        elif command == "ORDER":
            customer_id = parts[1]
            item_count = int(parts[2])

            order: List[OrderItem] = []

            position = 3

            for _ in range(item_count):
                product_id = int(parts[position])
                quantity = int(parts[position + 1])

                order.append((product_id, quantity))
                position += 2

            # -----------------------------------------------------
            # Phase 1: Validate the complete order.
            # No stock is modified here.
            # -----------------------------------------------------
            order_is_valid = True

            for product_id, quantity in order:
                if product_id not in products:
                    order_is_valid = False
                    break

                if product_id in out_of_stock_ids:
                    order_is_valid = False
                    break

                if products[product_id]["stock"] < quantity:
                    order_is_valid = False
                    break

            # -----------------------------------------------------
            # Phase 2: Apply changes only if the entire order
            # is valid.
            # -----------------------------------------------------
            if not order_is_valid:
                print(
                    f"Order {customer_id} "
                    "Rejected: Insufficient Stock."
                )
                continue

            total_cost = 0.0

            for product_id, quantity in order:
                product = products[product_id]

                total_cost += product["price"] * quantity

            # Deduct stock after complete validation.
            for product_id, quantity in order:
                product = products[product_id]

                product["stock"] -= quantity

                if product["stock"] == 0:
                    out_of_stock_ids.add(product_id)

            print(
                f"Order {customer_id} Approved: "
                f"${total_cost:.2f}"
            )

        # ---------------------------------------------------------
        # RESTOCK
        # ---------------------------------------------------------
        elif command == "RESTOCK":
            product_id = int(parts[1])
            quantity = int(parts[2])

            products[product_id]["stock"] += quantity

            # Product is no longer out of stock.
            if product_id in out_of_stock_ids:
                out_of_stock_ids.remove(product_id)

            print("Restock Success")


# Provided sample input
sample_input_3 = """\
3
101 Headphones 50.00 10 3 audio tech wireless
102 Earphones 20.00 5 2 audio wired
103 Mouse 25.00 2 2 tech wireless
4
FILTER_AND tech wireless
ORDER C101 2 101 2 103 5
ORDER C101 2 101 2 103 2
RESTOCK 103 10
"""

process_warehouse(sample_input_3)

101 103
Order C101 Rejected: Insufficient Stock.
Order C101 Approved: $150.00
Restock Success


In [7]:
# Verification Test 1: Provided sample
print("===== TEST 1: PROVIDED SAMPLE =====")
process_warehouse(sample_input_3)


# Verification Test 2: FILTER_OR
filter_or_test = """\
3
101 Headphones 50.00 10 2 audio wireless
102 Mouse 25.00 5 1 tech
103 Keyboard 40.00 3 2 office wireless
3
FILTER_OR wireless
FILTER_OR audio tech
FILTER_OR gaming
"""

print("\n===== TEST 2: FILTER_OR =====")
process_warehouse(filter_or_test)


# Verification Test 3: Atomic order verification
# The order asks for enough stock of product 101,
# but product 102 does not have enough stock.
# Therefore product 101 must NOT be deducted.
atomic_test = """\
2
101 Laptop 100.00 5 1 tech
102 Mouse 20.00 1 1 tech
3
ORDER C01 2 101 2 102 5
ORDER C01 1 101 2
RESTOCK 102 5
"""

print("\n===== TEST 3: ATOMIC ORDER =====")
process_warehouse(atomic_test)

===== TEST 1: PROVIDED SAMPLE =====
101 103
Order C101 Rejected: Insufficient Stock.
Order C101 Approved: $150.00
Restock Success

===== TEST 2: FILTER_OR =====
101 103
101 102
No Match

===== TEST 3: ATOMIC ORDER =====
Order C01 Rejected: Insufficient Stock.
Order C01 Approved: $200.00
Restock Success


# Problem 04: Inverted Index & Boolean Information Retrieval Engine

## Design Approach

An inverted index is constructed using a dictionary:

Word -> Set of Document IDs

For example, if the word "python" occurs in documents 1 and 2:

"python" -> {1, 2}

A set is used for Document IDs because a document should appear only once
for a particular word, even if that word occurs multiple times.

Each document is sanitized by:
1. Converting all text to lowercase.
2. Stripping punctuation.

The master set U contains all active Document IDs.

Boolean queries are solved using set algebra:

AND term1 term2
    -> Set(term1) intersection Set(term2)

OR term1 term2
    -> Set(term1) union Set(term2)

DIFF term1 term2
    -> Set(term1) difference Set(term2)

The resulting Document IDs are sorted in ascending order before printing.

## Complexity Analysis

Let D be the number of documents and W be the total number of words
across all documents.

- Building the inverted index: O(W)
- AND query: O(min(A, B)) average
- OR query: O(A + B) average
- DIFF query: O(A) average
- Sorting query results: O(R log R)
- Space complexity: O(W), where W represents the total number of
  word-document relationships stored in the inverted index.

In [8]:
from typing import Dict, Set
import string


def sanitize_text(text: str) -> list[str]:
    """
    Convert text to lowercase and remove punctuation.

    Args:
        text: Original document text.

    Returns:
        A list of sanitized lowercase words.
    """
    translation_table = str.maketrans(
        "",
        "",
        string.punctuation
    )

    cleaned_text = text.lower().translate(translation_table)

    return cleaned_text.split()


def build_inverted_index(
    documents: Dict[int, str]
) -> Dict[str, Set[int]]:
    """
    Build an inverted index from document IDs and their text.

    Args:
        documents: Dictionary mapping DocumentID to document text.

    Returns:
        Dictionary mapping each word to the set of Document IDs
        containing that word.
    """
    inverted_index: Dict[str, Set[int]] = {}

    for document_id, text in documents.items():
        words = sanitize_text(text)

        # Using a set avoids adding the same document ID
        # multiple times for repeated words.
        unique_words = set(words)

        for word in unique_words:
            if word not in inverted_index:
                inverted_index[word] = set()

            inverted_index[word].add(document_id)

    return inverted_index


def process_search_engine(input_data: str) -> None:
    """
    Build an inverted index and process Boolean search queries.
    """
    lines = input_data.strip().splitlines()
    index = 0

    # Number of documents.
    number_of_documents = int(lines[index])
    index += 1

    documents: Dict[int, str] = {}
    active_documents: Set[int] = set()

    # Read documents.
    for _ in range(number_of_documents):
        parts = lines[index].split()

        document_id = int(parts[0])
        document_text = " ".join(parts[1:])

        documents[document_id] = document_text
        active_documents.add(document_id)

        index += 1

    # Build inverted index.
    inverted_index = build_inverted_index(documents)

    # Number of queries.
    number_of_queries = int(lines[index])
    index += 1

    for _ in range(number_of_queries):
        parts = lines[index].split()
        index += 1

        command = parts[0]
        term_1 = parts[1]
        term_2 = parts[2]

        # Get document sets for both terms.
        # If a word does not exist, use an empty set.
        set_1 = inverted_index.get(term_1, set())
        set_2 = inverted_index.get(term_2, set())

        if command == "AND":
            result = set_1 & set_2

        elif command == "OR":
            result = set_1 | set_2

        elif command == "DIFF":
            result = set_1 - set_2

        else:
            result = set()

        sorted_result = sorted(result)

        if sorted_result:
            print(" ".join(map(str, sorted_result)))
        else:
            print("No Documents Found")


# Provided sample input
sample_input_4 = """\
3
1 python data structures and algorithms
2 python machine learning and neural networks
3 database management systems and sql queries
3
AND python algorithms
OR machine sql
DIFF python machine
"""

process_search_engine(sample_input_4)

1
2 3
1


In [9]:
# Verification Test 1: Provided sample
print("===== TEST 1: PROVIDED SAMPLE =====")
process_search_engine(sample_input_4)


# Verification Test 2: Case-insensitive and punctuation handling
punctuation_test = """\
3
1 Python, Python! DATA.
2 data science with Python.
3 JAVA programming
4
AND python data
OR java python
DIFF python data
AND python missing
"""

print("\n===== TEST 2: SANITIZATION =====")
process_search_engine(punctuation_test)


# Verification Test 3: No matching documents
empty_result_test = """\
2
1 apple banana
2 orange mango
2
AND apple orange
DIFF apple apple
"""

print("\n===== TEST 3: EMPTY RESULTS =====")
process_search_engine(empty_result_test)

===== TEST 1: PROVIDED SAMPLE =====
1
2 3
1

===== TEST 2: SANITIZATION =====
1 2
1 2 3
No Documents Found
No Documents Found

===== TEST 3: EMPTY RESULTS =====
No Documents Found
No Documents Found


# Problem 05: Continuous Activity Streak & Anomaly Cluster Detection

## Design Approach

A dictionary is used to store user activity:

Username -> List of timestamps

The list stores every login timestamp recorded for each user.

For longest consecutive streak detection, the user's timestamps are
converted into a Set.

For every timestamp x, we check whether x - 1 exists in the set.

- If x - 1 exists, x is not the beginning of a streak.
- If x - 1 does not exist, x is the beginning of a streak.
- Starting from x, we repeatedly check x + 1, x + 2, etc.

This avoids sorting the timestamps and gives O(n) average time for
longest consecutive sequence detection.

For Top-K ranking, users are sorted according to:
1. Total login count in descending order.
2. Longest streak in descending order.
3. Username in alphabetical order.

## Complexity Analysis

Let M be the total number of login events and U be the number of unique
users.

- Building the activity dictionary: O(M)
- Converting timestamps to sets: O(M)
- Longest streak detection for all users: O(M) average
- Ranking users: O(U log U)
- Overall time complexity: O(M + U log U)
- Space complexity: O(M + U)

The streak algorithm avoids sorting timestamps, which would require
O(n log n) time.

In [10]:
from typing import Dict, List, Tuple


def longest_consecutive_streak(timestamps: List[int]) -> int:
    """
    Find the longest consecutive sequence of timestamps.

    The algorithm uses a set and does not sort the timestamps.

    Args:
        timestamps: List of login timestamps.

    Returns:
        Length of the longest consecutive timestamp sequence.
    """
    if not timestamps:
        return 0

    timestamp_set = set(timestamps)
    longest_streak = 0

    for timestamp in timestamp_set:

        # A timestamp is a starting point only when
        # the previous second is not present.
        if timestamp - 1 not in timestamp_set:

            current_timestamp = timestamp
            current_streak = 1

            # Count consecutive timestamps.
            while current_timestamp + 1 in timestamp_set:
                current_timestamp += 1
                current_streak += 1

            longest_streak = max(
                longest_streak,
                current_streak
            )

    return longest_streak


def process_activity_logs(input_data: str) -> None:
    """
    Process login events and print user activity and Top-K users.
    """
    lines = input_data.strip().splitlines()
    index = 0

    # M = total login events
    # K = number of users to report
    total_events, top_k = map(int, lines[index].split())
    index += 1

    # Username -> list of timestamps
    user_logs: Dict[str, List[int]] = {}

    # Read login events.
    for _ in range(total_events):
        username, timestamp = lines[index].split()
        index += 1

        timestamp = int(timestamp)

        if username not in user_logs:
            user_logs[username] = []

        user_logs[username].append(timestamp)

    # Store:
    # (username, login_count, longest_streak)
    user_statistics: List[Tuple[str, int, int]] = []

    # Print every user alphabetically.
    for username in sorted(user_logs):
        timestamps = user_logs[username]

        login_count = len(timestamps)
        streak = longest_consecutive_streak(timestamps)

        user_statistics.append(
            (username, login_count, streak)
        )

        print(
            f"User: {username}, "
            f"Logins: {login_count}, "
            f"LongestStreak: {streak}"
        )

    # Sort for Top-K:
    # 1. Login count descending
    # 2. Streak descending
    # 3. Username ascending
    user_statistics.sort(
        key=lambda user: (
            -user[1],
            -user[2],
            user[0]
        )
    )

    print(f"\n-- Top {top_k} High-Activity Users --")

    for rank, (username, count, streak) in enumerate(
        user_statistics[:top_k],
        start=1
    ):
        print(
            f"Rank {rank}: {username} "
            f"({count} logins, streak: {streak})"
        )


# Provided sample input
sample_input_5 = """\
8 2
alice 100
bob 5
alice 4
alice 200
alice 102
alice 101
bob 6
bob 7
"""

process_activity_logs(sample_input_5)

User: alice, Logins: 5, LongestStreak: 3
User: bob, Logins: 3, LongestStreak: 3

-- Top 2 High-Activity Users --
Rank 1: alice (5 logins, streak: 3)
Rank 2: bob (3 logins, streak: 3)


In [11]:
# Verification Test 1: Provided sample
print("===== TEST 1: PROVIDED SAMPLE =====")
process_activity_logs(sample_input_5)


# Verification Test 2: Single timestamp
single_event_test = """\
3 1
alice 100
bob 500
alice 200
"""

print("\n===== TEST 2: SINGLE-EVENT STREAKS =====")
process_activity_logs(single_event_test)


# Verification Test 3: Tie-breaking
# Alice and Bob have the same login count and streak.
# Alphabetically, alice should rank before bob.
tie_test = """\
6 2
bob 10
bob 11
bob 12
alice 20
alice 21
alice 22
"""

print("\n===== TEST 3: TIE-BREAKING =====")
process_activity_logs(tie_test)

===== TEST 1: PROVIDED SAMPLE =====
User: alice, Logins: 5, LongestStreak: 3
User: bob, Logins: 3, LongestStreak: 3

-- Top 2 High-Activity Users --
Rank 1: alice (5 logins, streak: 3)
Rank 2: bob (3 logins, streak: 3)

===== TEST 2: SINGLE-EVENT STREAKS =====
User: alice, Logins: 2, LongestStreak: 1
User: bob, Logins: 1, LongestStreak: 1

-- Top 1 High-Activity Users --
Rank 1: alice (2 logins, streak: 1)

===== TEST 3: TIE-BREAKING =====
User: alice, Logins: 3, LongestStreak: 3
User: bob, Logins: 3, LongestStreak: 3

-- Top 2 High-Activity Users --
Rank 1: alice (3 logins, streak: 3)
Rank 2: bob (3 logins, streak: 3)
